In [0]:
print("hello mask")


In [0]:
# Bronze to Silver: Data Quality Check for Customer-Account Relationship

# Define your table names
bronze_customers_table = "bronze.customers"  # Update with your actual table name
bronze_accounts_table = "bronze.accounts"    # Update with your actual table name
silver_accounts_table = "silver.accounts"
rejected_accounts_table = "silver.rejected_accounts"  # For orphaned records

# Read bronze tables
df_customers = spark.table(bronze_customers_table)
df_accounts = spark.table(bronze_accounts_table)

# Find orphaned accounts (accounts without matching customer_id)
# Using left_anti join - keeps only records from accounts that DON'T have a match in customers
df_orphaned_accounts = df_accounts.alias("a").join(
    df_customers.alias("c"),
    df_accounts["customer_id"] == df_customers["customer_id"],  # Update join key if different
    "left_anti"
)

print(f"Found {df_orphaned_accounts.count()} orphaned accounts without matching customers")

# Show sample of orphaned records
if df_orphaned_accounts.count() > 0:
    print("\nSample orphaned accounts:")
    display(df_orphaned_accounts.limit(10))

# Find valid accounts (accounts WITH matching customers)
df_valid_accounts = df_accounts.alias("a").join(
    df_customers.alias("c"),
    df_accounts["customer_id"] == df_customers["customer_id"],
    "inner"  # Keep only accounts with valid customer references
).select("a.*")  # Select only account columns

print(f"\nFound {df_valid_accounts.count()} valid accounts with matching customers")

# Save orphaned accounts to rejected table for investigation
df_orphaned_accounts.write.mode("overwrite").saveAsTable(rejected_accounts_table)
print(f"\nSaved orphaned accounts to: {rejected_accounts_table}")

# Save valid accounts to silver layer
df_valid_accounts.write.mode("overwrite").saveAsTable(silver_accounts_table)
print(f"Saved valid accounts to: {silver_accounts_table}")

print("\n✅ Data quality check completed!")